In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# --- 1. Load and Inspect Dataset ---
DATA_PATH = "talabat_enhanced_orders.csv"
df = pd.read_csv(DATA_PATH)

# --- 2. Original Feature Engineering ---
df_fe = df.copy()

# Time-based features
df_fe["Order_Time"] = pd.to_datetime(df_fe["Order_Time"], errors="coerce")
df_fe["order_hour"] = df_fe["Order_Time"].dt.hour
df_fe["order_dayofweek"] = df_fe["Order_Time"].dt.dayofweek
df_fe["is_weekend"] = df_fe["order_dayofweek"].isin([5,6]).astype(int)

# Original peak-hour rule
df_fe["is_peak_hour"] = df_fe["order_hour"].isin(list(range(12,16)) + list(range(19,24))).astype(int)

# Price-based features
df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]

# GPS Distance (Haversine)
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df_fe["haversine_rest_to_cust_km"] = haversine_km(
    df_fe["Restaurant_Lat"], df_fe["Restaurant_Lon"],
    df_fe["Customer_Lat"], df_fe["Customer_Lon"]
)

# Item cardinality reduction
top_k = 20
top_items = df_fe["Item_Name"].value_counts().head(top_k).index
df_fe["Item_Name_reduced"] = np.where(df_fe["Item_Name"].isin(top_items), df_fe["Item_Name"], "Other")

# Binning
df_fe["price_tier"] = pd.cut(
    df_fe["Total_Price"],
    bins=[0, 100, 250, 500, np.inf],
    labels=["low","medium","high","very_high"]
)

# --- 3. Data Preparation ---
target_col = "Order_Status"
drop_cols = [
    "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
    "Order_Time", "Delivery_Time", "Delivery_Duration_Minutes", "Item_Name"
]
X = df_fe.drop(columns=drop_cols + [target_col])
y = df_fe[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 4. Baseline Modeling ---
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"))
])

model.fit(X_train, y_train)

# ==========================================
# STUDENT TASKS IMPLEMENTATION
# ==========================================

# --- Task 1: New Engineered Feature ---
# Justification: High quantity paired with high total price often indicates large bulk or catering orders.
# These may face higher cancellation rates due to prep complexity.
df_fe["quantity_price_interaction"] = df_fe["Quantity"] * df_fe["Total_Price"]

# --- Task 2: Different Peak Hour Rule ---
# Adding morning rush hour (7-9 AM) to the existing lunch and dinner rules.
df_fe["is_peak_hour_v2"] = df_fe["order_hour"].isin(
    list(range(7, 10)) + list(range(12, 16)) + list(range(19, 24))
).astype(int)

# --- Task 3: Varying top_k for Item_Name_reduced ---
# Example with top_k = 10
top_k_10 = 10
top_items_10 = df_fe["Item_Name"].value_counts().head(top_k_10).index
df_fe["Item_Name_reduced_k10"] = np.where(df_fe["Item_Name"].isin(top_items_10), df_fe["Item_Name"], "Other")

# --- Task 4: Model-Based Feature Selection ---
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", selector),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"))
])

# To execute the new task pipeline:
model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Accuracy with Feature Selection:", round(accuracy_score(y_test, y_pred_fs), 4))
print("\nClassification Report with Feature Selection:")
print(classification_report(y_test, y_pred_fs))

KeyboardInterrupt: 